# Context Windows & Prompt Caching

**Goal:** Budget the context window, use prompt caching, and compare batch vs real-time pricing.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks) — the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).


In [ ]:
%pip install -q anthropic

In [ ]:
import os

# In Colab, read the key from Secrets (key icon in the left sidebar).
# Locally, set the ANTHROPIC_API_KEY env var instead.
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except ImportError:
    assert os.environ.get('ANTHROPIC_API_KEY'), 'Set ANTHROPIC_API_KEY'

import anthropic
client = anthropic.Anthropic()
MODEL = 'claude-sonnet-5'  # good default: capable and cheap enough to iterate on


## The mental model: a per-request token budget

Every request spends from one budget: the context window (1M tokens on current Sonnet/Opus models — but you'll rarely want to fill it; cost and latency scale with what you send). Think of each request as a ledger with four lines:

| Line | Typical share | Notes |
|---|---|---|
| System prompt | fixed, small-ish | instructions, persona, tool definitions |
| Retrieved context | the big variable | RAG chunks, documents, code |
| Conversation history | grows every turn | the thing that eventually eats everything |
| Output headroom | reserve it! | `max_tokens` counts against the window |

The failure mode isn't hitting the hard limit — it's cost creep: an agent that resends 50k tokens of history to answer a 10-token question. Budgeting means measuring each line, and the API gives you an exact meter: `count_tokens`. It's free and model-specific — never estimate Claude tokens with `tiktoken` (that's OpenAI's tokenizer, and it's wrong for Claude by a lot).


In [ ]:
def count(messages, system=None):
    kwargs = {'model': MODEL, 'messages': messages}
    if system is not None:
        kwargs['system'] = system
    return client.messages.count_tokens(**kwargs).input_tokens


SYSTEM = 'You are a support assistant for AcmeDB. Answer only from the provided runbook. Cite section numbers.'

print('system + 1 short question:',
      count([{'role': 'user', 'content': 'How do I rotate credentials?'}], system=SYSTEM))

# Same content, two phrasings — tokens are about bytes-of-text, not 'ideas'
terse = 'rotate creds acmedb how'
prose = ('Hello! I was wondering if you could kindly walk me through the process of '
         'rotating the credentials for our AcmeDB instance when you get a chance?')
print('terse phrasing: ', count([{'role': 'user', 'content': terse}]))
print('prose phrasing: ', count([{'role': 'user', 'content': prose}]))


In [ ]:
# A budget checker you can keep: does this request fit the plan?
def check_budget(system, context, history, max_output,
                 budget={'system': 2_000, 'context': 8_000, 'history': 4_000}):
    # 3 count_tokens calls (free), no generation.
    lines = {
        'system':  count([{'role': 'user', 'content': 'x'}], system=system) ,
        'context': count([{'role': 'user', 'content': context}]),
        'history': count(history) if history else 0,
    }
    for name, used in lines.items():
        cap = budget[name]
        flag = 'OK ' if used <= cap else 'OVER'
        print(f'{flag} {name:8s} {used:>7,} / {cap:,}')
    total = sum(lines.values()) + max_output
    print(f'    total incl. {max_output:,} output headroom: {total:,} tokens')
    return lines


_ = check_budget(
    system=SYSTEM,
    context='Section 4.2: To rotate credentials, run acmedb rotate --all ...' * 40,
    history=[{'role': 'user', 'content': 'hi'}, {'role': 'assistant', 'content': 'Hello! How can I help?'}],
    max_output=1_000,
)


## Prompt caching

If every request re-sends the same big system prompt or document, you're paying full price to re-process bytes the model saw seconds ago. Prompt caching fixes that: mark a breakpoint with `cache_control`, and the API caches the processed prefix.

The rules that matter:

- **It's a prefix match on exact bytes.** The request renders as `tools` → `system` → `messages`; any byte change *before* your breakpoint invalidates everything after it.
- **Economics:** a cache *write* costs ~1.25x the input rate; a cache *read* costs ~0.1x. Break-even after the second request.
- **TTL is 5 minutes** (refreshed on each hit; a 1-hour tier exists at 2x write cost).
- **Minimum cacheable prefix** is a few thousand tokens depending on model — short prompts silently don't cache (no error, just `cache_creation_input_tokens: 0`).

Let's prove it with a long fake runbook: call twice, watch the usage fields flip from *creation* to *read*.


In [ ]:
# Build a deterministic ~8k-token 'runbook' (deterministic matters: caching is byte-exact).
sections = []
for i in range(1, 121):
    sections.append(
        f'Section {i}: Procedure AC-{i:03d}. To perform maintenance task {i}, first '
        f'verify replica lag is under {i * 5} ms, then drain node group {i % 7}, apply '
        f'the configuration bundle, and re-enable traffic. Rollback: restore snapshot '
        f'tagged ac-{i:03d}-pre and replay the WAL from the checkpoint marker.'
    )
RUNBOOK = '\n'.join(sections)

print('runbook tokens:',
      count([{'role': 'user', 'content': RUNBOOK}]))


In [ ]:
def ask_runbook(question):
    return client.messages.create(
        model=MODEL,
        max_tokens=150,
        system=[{
            'type': 'text',
            'text': 'Answer from this runbook only, citing section numbers.\n\n' + RUNBOOK,
            'cache_control': {'type': 'ephemeral'},  # breakpoint: cache everything up to here
        }],
        messages=[{'role': 'user', 'content': question}],
    )


# 2 API calls. Run them within 5 minutes of each other (the cache TTL).
for question in ['What is the rollback for procedure AC-042?',
                 'Which node group does task 10 drain?']:
    r = ask_runbook(question)
    u = r.usage
    print(f'Q: {question}')
    print(f'   input={u.input_tokens}  cache_creation={u.cache_creation_input_tokens}  '
          f'cache_read={u.cache_read_input_tokens}')
    print(f'   A: {r.content[0].text[:110]}...')
    print()


Run it and note the flip: call 1 has a large `cache_creation_input_tokens` (you paid the 1.25x write), call 2 has a large `cache_read_input_tokens` (you paid 0.1x) and a tiny `input_tokens` — just the new question. That ~90% discount on the big prefix is the whole feature.

### What breaks the cache

All of these silently zero your hit rate — the request still works, you just pay full price:

- A timestamp, request ID, or user name interpolated into the system prompt — the prefix differs every request. Put volatile content in `messages`, *after* the breakpoint.
- Adding, removing, or reordering **tools** — tools render at position 0, ahead of system. Serialize them deterministically.
- Switching **models** mid-conversation — caches are per-model.
- Non-deterministic serialization (`json.dumps` without `sort_keys`, iterating a set).

Debug rule: if `cache_read_input_tokens` stays 0 across identical-looking requests, diff the rendered bytes of two requests — something is varying.


## The Batch API: half price for patience

Everything above is real-time. The Batches API takes the same request payloads, processes them asynchronously (most batches finish well under an hour, 24h worst case), and charges **50%** of standard price.

In FDE/AI-engineer work this maps to a specific set of jobs: **bulk evals** (re-score 500 transcripts against a new judge prompt), **backfills** (classify every historical ticket with the new taxonomy), nightly enrichment, regression sweeps before a release. The tell is always the same — nobody is waiting on the output, and there are a lot of them. If a human is watching a spinner, batch is wrong; if it's a cron job, batch is free money.

The sketch below submits a tiny 3-request batch on Haiku (bulk tier — that's the point) and polls briefly. Small batches usually finish in a couple of minutes; if it's still processing when the poll loop gives up, re-run the results cell later.


In [ ]:
import time
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request

reviews = [
    'Battery died after two days. Never again.',
    'Does exactly what it says. Five stars.',
    'Fine I guess. Packaging was nice.',
]

batch = client.messages.batches.create(
    requests=[
        Request(
            custom_id=f'review-{i}',
            params=MessageCreateParamsNonStreaming(
                model='claude-haiku-4-5',   # bulk work -> cheap tier, batched -> half of cheap
                max_tokens=10,
                messages=[{'role': 'user',
                           'content': f'Sentiment, one word (positive/negative/neutral): {text}'}],
            ),
        )
        for i, text in enumerate(reviews)
    ]
)
print('batch id:', batch.id, '| status:', batch.processing_status)


In [ ]:
# Poll briefly (up to ~2 minutes). If it's still running, just re-run this cell later.
for _ in range(12):
    batch = client.messages.batches.retrieve(batch.id)
    if batch.processing_status == 'ended':
        break
    print('still', batch.processing_status, '...')
    time.sleep(10)

if batch.processing_status == 'ended':
    # Results arrive in ARBITRARY order — always key by custom_id, never by position.
    for result in client.messages.batches.results(batch.id):
        if result.result.type == 'succeeded':
            msg = result.result.message
            text = next((b.text for b in msg.content if b.type == 'text'), '')
            print(f'{result.custom_id}: {text.strip()}')
        else:
            print(f'{result.custom_id}: {result.result.type}')
else:
    print('not done yet — re-run this cell in a minute or two')


## Napkin math you should be able to do cold

Every pricing conversation reduces to `tokens x rate`, with three multipliers: cache write ~1.25x input, cache read ~0.1x input, batch 0.5x everything. An AI engineer who can't sketch this on a napkin will get surprised by a bill; one who can will spot the 10x saving in a design review. Helper below, then two worked examples.


In [ ]:
PRICES = {
    # USD per million tokens: (input, output). As of mid-2026 — these drift; update from the pricing page.
    'claude-sonnet-5':  (3.00, 15.00),
    'claude-haiku-4-5': (1.00, 5.00),
    'claude-opus-4-8':  (5.00, 25.00),
}


def estimate_cost(model, input_tokens=0, output_tokens=0,
                  cache_read=0, cache_write=0, batch=False):
    inp, out = PRICES[model]
    usd = (input_tokens * inp
           + output_tokens * out
           + cache_read * inp * 0.10
           + cache_write * inp * 1.25) / 1_000_000
    return usd * (0.5 if batch else 1.0)


# Example 1: a support bot. 10k requests/day, 8k-token runbook prefix, 200-token
# question, 300-token answer. Compare no-cache vs cached prefix.
n = 10_000
no_cache = n * estimate_cost('claude-sonnet-5', input_tokens=8_200, output_tokens=300)
cached = (estimate_cost('claude-sonnet-5', cache_write=8_000, input_tokens=200, output_tokens=300)
          + (n - 1) * estimate_cost('claude-sonnet-5', cache_read=8_000, input_tokens=200, output_tokens=300))
print(f'support bot / day  no cache: ${no_cache:,.2f}   cached: ${cached:,.2f}   '
      f'saving: {1 - cached / no_cache:.0%}')

# Example 2: backfill-classify 1M tickets (500 in / 10 out each) on Haiku, batch vs realtime.
rt = 1_000_000 * estimate_cost('claude-haiku-4-5', input_tokens=500, output_tokens=10)
bt = 1_000_000 * estimate_cost('claude-haiku-4-5', input_tokens=500, output_tokens=10, batch=True)
print(f'1M-ticket backfill  realtime: ${rt:,.2f}   batch: ${bt:,.2f}')

# And the wrong-model version of example 2, because someone always proposes it:
opus_rt = 1_000_000 * estimate_cost('claude-opus-4-8', input_tokens=500, output_tokens=10)
print(f'same backfill on realtime opus: ${opus_rt:,.2f}  <- this is the design-review catch')


Run it and note the shape of the savings: caching attacks the *repeated prefix* (dominant when the shared context is big), batch attacks *everything* (dominant for bulk jobs), and model choice dwarfs both. The three multiply — Haiku + batch + a cached prefix is routinely 20-50x cheaper than realtime Opus with no cache, for work where the cheap version passes the same eval.


## Exercises

1. Measure the cache TTL empirically: run `ask_runbook` twice back-to-back, wait 6+ minutes, run it again, and compare the usage fields of all three calls.
2. Break the cache on purpose: add `f'Today is {datetime.now()}. '` to the front of the system text and make two calls. Confirm `cache_read_input_tokens` stays 0, then move the timestamp into the user message and confirm the cache works again.
3. Write `fit_history(history, budget_tokens)` that drops the oldest turns (always in user/assistant pairs) until `count_tokens` says the history fits the budget. Test it on a fabricated 20-turn conversation.
4. Extend `estimate_cost` to take a `calls_per_day` and `cache_hit_rate` and produce a monthly projection table for all three models, cached and uncached. Sanity-check one row by hand.
